In [1]:
from platform import python_version
print(python_version())

3.12.12


### A cross-scale multimodal framework identifies clinically actionable immunotherapy biomarkers in melanoma (study, forked)

Huoshen W, Yuan K, Xiong J, Lin Y, Yu W, Xie Y, Yuan Q, Zhang X, Dong C, Sun C, Yi S. A cross-scale multimodal framework identifies clinically actionable immunotherapy biomarkers in melanoma through bulk to single-cell and spatial transcriptomics integration. Hum Genomics. 2026 Mar 16;20(1):77. doi: 10.1186/s40246-026-00919-w. PMID: 41840725; PMCID: PMC13104449.

https://pubmed.ncbi.nlm.nih.gov/41840725/

In [2]:
import os, sys, re, yaml
from pathlib import Path
from dotenv import load_dotenv
import time
import requests
import json

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

from scipy.stats import spearmanr

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

ROOT0 = Path("/home/flavio/uv/rna_velocity")
ROOT_SRC = ROOT0 / "src"

sys.path.insert(0, str(ROOT_SRC))

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import create_dir, pdreadcsv, pdwritecsv

root_figure = create_dir(ROOT0, "pictures")
root_results = create_dir(ROOT0, "results")
root_tables = create_dir(root_results, "tables")

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/rna_velocity
ROOT_SRC added: /home/flavio/uv/rna_velocity/src


In [3]:
root_search = create_dir(ROOT0, "download/search")
os.listdir(root_search)

['paper_data_availability_spatial_scvelo.csv']

### Start with GSE148190 (Apr 07, 2020)

https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE148190

**title**: Single cell RNA and TCR sequencing of tumor-infiltrating lymphocytes from human melanoma

**reference**: Mahuron KM, Moreau JM, Glasgow JE, Boda DP, Pauli ML, Gouirand V, Panjabi L, Grewal R, Luber JM, Mathur AN, Feldman RM, Shifrut E, Mehta P, Lowe MM, Alvarado MD, Marson A, Singer M, Wells J, Jupp R, Daud AI, Rosenblum MD. Layilin augments integrin activation to promote antitumor immunity. J Exp Med. 2020 Sep 7;217(9):e20192080. doi: 10.1084/jem.20192080. PMID: 32539073; PMCID: PMC7478725.

**features**: human, melanoma, n_samples=14, and scRNA 10x 5' GEX + VDJ

**platform**: GPL24676	Illumina NovaSeq 6000 (Homo sapiens)

https://seqout.org/p/SRP255479

### GSE148190: Single cell RNA and TCR sequencing of (human) tumor-infiltrating lymphocytes from human melanoma

https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE148190

Tasks using Python and scVelo, download GSE148190 data

### Plan (reviwed by Claude-Science Opus 5)

1. Step 1 cannot feed step 6. GEO gives you CellRanger matrices — **spliced counts** only. **scVelo needs an unspliced layer**, and GSE148190 deposits no loom/velocyto output and no BAMs. So **"download GSE148190" has to mean download FASTQ and re-quantify with intron-aware counting** (kb count --workflow nac), not download the matrices. That single point reorders the whole pipeline.

2. Scope has to be decided before downloading, not after. **All six GEX libraries are 1.15 Tbases** (~279 GB for the three K409 ones alone). There's no remote compute configured and 444 GB of local disk, so I scoped it to patient K409 — cutaneous primary tumour, involved lymph node, blood — two of four lanes each (140 GB, ~360M read pairs/library, which saturates a 10x channel). One patient across three sites also removes the cross-patient batch effect from the velocity step entirely.

3. "QC we already have" — we don't. Everything so far has been metadata curation. No cell has been counted yet, so mito%, complexity, doublets and the unspliced fraction all still need to be computed, and the unspliced fraction is a QC axis that only exists after re-quantification.

4. "PCA correlation checks": variance-explained elbow plus Spearman correlation of PC1–30 against total counts, genes, mito%, unspliced fraction and library/tissue — the output being a decision on whether the neighbour graph needs **Harmony correction**. That decision has to be made before UMAP/Leiden, because the same graph is reused for scVelo's moments.

5. Top-6 HVGs is a weak fifth step. In FACS-sorted lymphocytes the dispersion-ranked genes are routinely ribosomal, mitochondrial, heat-shock or clonal TCR V-segments, i.e. not cell-state markers. I kept your panel but paired it with canonical T-cell markers so the comparison is visible rather than assumed.
Velocity needs a falsifiable expectation. These are terminally differentiated/exhausted T cells, where splicing-kinetics assumptions are weakest. Stated up front: arrows should run naive/memory → effector → exhausted (the source paper's LAYN⁺ population being the terminal state). If the field is incoherent or confidence is low, that gets reported as the result rather than tuned until it looks plausible.

6. Missing from the original: doublet detection, a check that the sort actually excluded non-T cells, cycling cells as a competing velocity axis, an h5ad checkpoint between stages, and velocity diagnostics (confidence, latent time, PAGA) rather than just a stream plot.

Two things you should know as facts about the data rather than choices: 
  1. velocity here will be on T cells only — every library is CD45⁺CD3⁺ sorted, 
  2. so no malignant melanoma cells exist in this dataset — and 
  3. this series carries no treatment-response labels, so the responder/non-responder axis from the earlier filtering is not available in GSE148190.

Downloads are running now (140 GB, ~3 h at the throughput ENA is giving); the Python 3.12 environment is building in parallel.

Next, implement using Python 3.12

### GSE148190 — RNA velocity in patient K409 T cells across tumour, lymph node and blood

Re-quantification of three 10x 5' GEX libraries from patient K409 (Mahuron et al.,
*J Exp Med* 2020, PMID 32539073) to spliced/unspliced counts, followed by QC, PCA
diagnostics, clustering and scVelo. GEO deposits CellRanger matrices only, so the
unspliced layer had to be rebuilt from raw reads.

## Data and quantification

| library | GSM | site | reads | pseudoaligned | cells called | median UMI | unspliced reads |
|---|---|---|---|---|---|---|---|
| tumor_primary | GSM4455935 | cutaneous primary tumour | 181.4 M | 70.9% | 2,337 | 3,190 | 12.1% |
| lymph_node | GSM4455933 | involved regional lymph node | 164.5 M | 76.7% | 6,511 | 3,366 | 7.2% |
| blood_PBMC | GSM4455932 | peripheral blood | 186.0 M | 78.8% | 5,494 | 3,627 | 7.7% |

One of four sequencing lanes per library (69.9 GB of FASTQ, md5-verified against the ENA
filereport) — ~180 M read pairs per channel, which saturates a 10x library at these cell
numbers.

**kb-python 0.30.2**, `kb count --workflow nac`, prebuilt human nascent/mature index,
`-x 10XV2` (16 bp barcode + 10 bp UMI, the 5' v1.1 geometry), `--strand reverse`.
Strandedness was determined empirically rather than assumed: on 1 M read pairs, `reverse`
gave 70.8% pseudoalignment against 18.1% for `forward` (78.6% unstranded). Layers:
`spliced` = mature, `unspliced` = nascent, `ambiguous` kept separately, `X` = mature +
ambiguous.

**Cells** are the authors' CellRanger barcode lists from GEO (all present in the kb
output). The log-log knee is stricter (1663/5314/4378 cells) and every knee barcode is a subset
of the author call, so the published cell set was used for comparability.

## QC

| library       |   cells_in |   fail_genes |   fail_umi |   fail_mito |   doublets |   cells_out |   median_umi |   median_genes |   median_pct_mt |   median_unspliced_frac |
|:--------------|-----------:|-------------:|-----------:|------------:|-----------:|------------:|-------------:|---------------:|----------------:|------------------------:|
| tumor_primary |       2337 |            2 |          3 |          45 |          1 |        2289 |         3190 |           1023 |         2.15548 |               0.113442  |
| lymph_node    |       6511 |            0 |          0 |          36 |          0 |        6475 |         3366 |           1085 |         2.9552  |               0.0697674 |
| blood_PBMC    |       5494 |            0 |          6 |           8 |          1 |        5483 |         3627 |           1012 |         4.47106 |               0.0755567 |

14,342 called cells → **14,247 cells × 18,623 genes** (>=200 genes, >=500 UMI, <=10%
mitochondrial, non-doublet). Losses are almost entirely mitochondrial-high tumour cells.
Scrublet flagged 2 doublets in total; on FACS-sorted lymphocytes simulated doublets
resemble real cells, so this is treated as uninformative rather than as evidence of a
doublet-free dataset.

## PCA diagnostics

2,000 HVGs (seurat_v3 on counts, `batch_key="library"`), 50 PCs, 30 used
(39.6% of variance). PC1 correlates with percent-ribosomal at
rho = -0.86 and with the blood indicator at rho = -0.64; no PC
beyond PC5 exceeds |rho| = 0.4 on any technical covariate.

**No batch correction was applied.** The three libraries are three anatomical sites from ONE patient, prepared with the same 10x 5' chemistry; 'library' is therefore confounded with tissue of origin, which is the biology under study. PC1 loads on ribosomal content (rho=-0.86) and separates blood (rho=-0.64) from tumour/node, i.e. the naive-to-activated axis. Harmony over 'library' would remove exactly that signal, so the uncorrected PCA is used and no batch correction is applied.

## Clusters

Leiden at 0.4 / 0.8 / 1.2 → 9 / 14 / 15 clusters; resolution 0.8 used. Labels come from
mean marker expression per cluster, not module-score argmax: *CD4* mRNA is near-absent
(<=0.42 in every cluster), and the automated version mislabelled the entire blood CD8
compartment as CD4. Blood clusters are uniformly *CD8A/CD8B*+, matching the CD8-only sort
recorded for GSM4455932.

| annotation            |   cutaneous primary tumour |   involved regional lymph node |   peripheral blood |
|:----------------------|---------------------------:|-------------------------------:|-------------------:|
| CD8 cycling exhausted |                         71 |                            179 |                  0 |
| CD4 Treg              |                        203 |                            669 |                  0 |
| naive/CM T            |                         49 |                            952 |                  1 |
| CD8 GZMK+ EM          |                        347 |                            524 |                  4 |
| CD8 exhausted         |                        768 |                            579 |                  9 |
| CD8 cytotoxic PD-1hi  |                        394 |                            924 |                 11 |
| CD4 memory            |                        212 |                           1483 |                  8 |
| CD8 EM GNLY+          |                        206 |                            857 |                  7 |
| B cell                |                         35 |                            301 |                  5 |
| CD8 TEMRA (blood)     |                          1 |                              2 |               1317 |
| CD8 naive (blood)     |                          2 |                              4 |               1520 |
| CD8 memory (blood)    |                          1 |                              1 |               1137 |
| CD8 GZMK+ EM (blood)  |                          0 |                              0 |               1271 |
| CD8 effector (blood)  |                          0 |                              0 |                193 |

The 341 *MS4A1*/*CD79A*+, *CD3D*-negative cells are B-cell contamination of the sort,
concentrated in the node.

## RNA velocity

Stochastic: 705 velocity genes, median confidence
0.72. Dynamical: 701 genes fitted,
289 velocity genes, median confidence
0.69. *PDCD1*, *HAVCR2* and *LAYN* — the exhaustion genes
central to the source paper — were all successfully fitted.

The pre-registered expectation was that arrows run naive/memory -> effector -> exhausted.
Result: **partial support, and the estimators disagree.**

| test (Spearman rho) | all cells | tissue only | blood only |
|---|---|---|---|
| pseudotime vs exhaustion score | 0.573 | 0.538 | -0.148 |
| pseudotime vs naive score | -0.408 | -0.593 | -0.665 |
| latent time vs exhaustion score | -0.159 | 0.333 | 0.218 |
| pseudotime vs latent time | -0.37 | 0.482 | -0.671 |

- **Cell level, within compartment: consistent.** In tumour + node, velocity pseudotime
  rises with the exhaustion module (rho = +0.54) and falls with the naive module
  (rho = -0.59); the terminal cluster is CD8 cytotoxic PD-1hi (median pseudotime 0.96).
  In blood it runs from naive toward TEMRA (rho = -0.67 against the naive score).
- **Cluster level: inconsistent.** PAGA velocity-directed edges in the tissue compartment
  carry more weight *away* from exhaustion (0.55) than toward it
  (0.117); the strongest tissue edges are CD8 EM GNLY+ -> CD4 memory (0.15)
  and CD8 cytotoxic PD-1hi -> CD8 EM GNLY+ (0.12).
- **Dynamical latent time is unreliable here.** Blood naive cells carry the *highest*
  latent time (0.83) and latent time anti-correlates with pseudotime in blood
  (rho = -0.67), the expected failure when the manifold has disconnected components and a
  single global root is inferred.

Top likelihood-ranked drivers: *NKG7*, *LDHA*, *PRSS57*, *CA2*, *IL7R*, *AHI1*, *CLIC1*, *RPS13*. Note that
*PRSS57* and *CA2* have very few non-zero cells, so their high likelihood is an artefact of
sparse support rather than evidence of kinetics — visible directly in the phase portraits.

No parameters were retuned to improve the arrows.

## Caveats that bound interpretation

1. **One patient.** No cross-patient replication; K383/K411 and the 2025-added
   nivolumab-treated samples (K374, K415) were not processed — the latter are SRA status
   `hup` (embargoed).
2. **T cells only.** Every library is CD45+CD3+ sorted, so no malignant melanoma cells
   exist in this dataset; velocity describes lymphocyte state transitions.
3. **No treatment-response labels** in GSE148190; these samples are treatment-naive.
4. **Three anatomical sites analysed jointly.** Blood forms a manifold disjoint from
   tumour/node, so the velocity graph does not connect them and arrows must be read within
   compartment.
5. **Low unspliced fraction** (7.2-12.1% of reads; median 0.077 per cell) — at the low
   end for velocity, and 5' chemistry captures less intronic signal than 3'.
6. **Terminally differentiated cells.** Steady-state and splicing-kinetics assumptions are
   weakest exactly in exhausted/effector populations, the likely source of the
   cluster-level incoherence above.
7. **Heat-shock signature.** Four of the six top HVGs are chaperones — a dissociation
   stress response present in the original protocol, not regressed out here.
8. Assumption diagnostics beyond those reported (e.g. formal tests of the steady-state
   model) were not assessed.

## Reproduction

`load_qc.py` -> `step_qc.py` -> `step_pca.py` -> `step_cluster.py` -> `step_velocity.py`,
environment `velo-pinned` (Python 3.12.14, numpy 1.26.4, pandas 2.2.3, scanpy 1.11.5,
scvelo 0.3.4, anndata 0.12.19). scvelo 0.3.4 is **not** numpy-2 compatible: stochastic mode
raises in `leastsq_generalized` and dynamical mode hits read-only arrays, which is why the
pinned environment exists alongside the kb-python one. `recover_dynamics` and
`velocity_graph` must run with `n_jobs=1`, `backend="threading"`,
`show_progress_bar=False` because the sandbox blocks the AF_UNIX sockets
`multiprocessing.Manager` needs.

## Figures

![QC and cell calling]({{artifact:art_d5f18e22-145e-4c29-a18a-aacea2ccea02}})

![PCA diagnostics]({{artifact:art_feeea23c-5f78-49a7-9ede-f01fc63aa3c6}})

![UMAP and clusters]({{artifact:art_93e3ca51-37d5-47f5-b819-38c35ce7cc23}})

![Top HVGs vs canonical markers]({{artifact:art_d6a7d736-c06a-4785-bea9-fd1614acc348}})

![Velocity stream, confidence and time]({{artifact:art_d9778303-4834-48e9-ac97-619a07d5c518}})

![Phase portraits]({{artifact:art_23b5b328-d09a-44dd-94fd-322ed0918ced}})
